# Análise Comparativa de Desempenho: Ingestão Batch vs. Stream

**Workloads:** 1x · 2x · 4x &emsp; **Execuções por workload:** 3  
**Objetivo:** (1) verificar consistência intra-workload entre repetições; (2) comparar os paradigmas batch/stream entre workloads nas métricas de throughput, latência e recursos Docker.

## 0. Setup

In [ ]:
import json, glob, os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')

# ── Estilo acadêmico IEEE ────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':       'serif',
    'font.size':          9,
    'axes.titlesize':    10,
    'axes.labelsize':     9,
    'xtick.labelsize':    8,
    'ytick.labelsize':    8,
    'legend.fontsize':    8,
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.35,
    'grid.linestyle':    '--',
})

# Paleta Paul Tol – daltônica-segura
C = {'1x': '#4477AA', '2x': '#EE6677', '4x': '#228833',
     'batch': '#4477AA', 'stream': '#EE6677'}

# Tamanhos IEEE (polegadas)
FIG1 = (3.54, 2.6)   # coluna única
FIG2 = (7.16, 3.0)   # duas colunas
FIG2T= (7.16, 4.5)   # duas colunas, mais alto
FIG3 = (7.16, 5.5)   # tall

RUNS_DIR  = 'runs'
FIGS_DIR  = 'Figs/comparativo'
os.makedirs(FIGS_DIR, exist_ok=True)
print('Setup OK')

## 1. Carregamento de dados

In [ ]:
# ── Inventário de runs ────────────────────────────────────────────────────────
# (workload, run_label, run_dir, tem_summary_completo)
RUN_META = [
    ('1x', 'R1', 'runs/experimento_1x_2026_06_29_13_46', True),
    ('1x', 'R2', 'runs/experimento_1x_2026_06_29_16_19', False),  # docker stats only
    ('1x', 'R3', 'runs/experimento_1x_2026_06_29_17_01', True),
    ('2x', 'R1', 'runs/experimento_2x_2026_06_29_21_44', False),  # docker stats only
    ('2x', 'R2', 'runs/experimento_2x_2026_06_29_22_47', True),
    ('2x', 'R3', 'runs/experimento_2x_2026_06_29_23_57', True),
    # 4x R1: PG sem truncate (total_records inflado); throughput e docker válidos
    ('4x', 'R1', 'runs/experimento_4x_2026_06_10_03_34', True),
    ('4x', 'R2', 'runs/experimento_4x_2026_06_29_02_08', True),
    # experimento_4x_2026_06_29_02_07: metrics vazio – omitido
]
WORKLOADS = ['1x', '2x', '4x']

# ── Helpers ───────────────────────────────────────────────────────────────────
def parse_bytes_mib(s):
    m = re.match(r'([\d.]+)\s*(B|kB|MB|MiB|GiB|GB)', str(s).strip(), re.I)
    if not m: return np.nan
    v, u = float(m.group(1)), m.group(2).lower()
    return v * {'b':1/1048576,'kb':1/1024,'mb':1,'mib':1,'gib':1024,'gb':953.674}[u]

# ── Summaries ─────────────────────────────────────────────────────────────────
summaries = []
for wl, rid, rdir, has_sum in RUN_META:
    if not has_sum: continue
    jf = glob.glob(f'{rdir}/metrics/summary_*.json')
    if not jf: continue
    with open(jf[0], encoding='utf-8') as f:
        d = json.load(f)
    d.update({'workload': wl, 'run_id': rid, 'run_dir': rdir})
    summaries.append(d)

# ── Batch throughput ──────────────────────────────────────────────────────────
tp_frames = []
for wl, rid, rdir, has_sum in RUN_META:
    if not has_sum: continue
    tf = glob.glob(f'{rdir}/metrics/batch_throughput_*.csv')
    if not tf: continue
    df = pd.read_csv(tf[0]).drop_duplicates(subset=['batch_id'])
    df['workload'], df['run_id'] = wl, rid
    tp_frames.append(df)
df_tp = pd.concat(tp_frames, ignore_index=True)

# ── Docker stats ──────────────────────────────────────────────────────────────
dock_frames = []
for wl, rid, rdir, _ in RUN_META:
    df = glob.glob(f'{rdir}/metrics/docker_stats_*.csv')
    if not df: continue
    d = pd.read_csv(df[0])
    d['workload'], d['run_id'] = wl, rid
    d['timestamp'] = pd.to_datetime(d['timestamp'], utc=True, errors='coerce')
    d['cpu_pct'] = d['cpu_perc'].str.rstrip('%').astype(float)
    d['mem_mib'] = d['mem_usage'].str.split('/').str[0].apply(parse_bytes_mib)
    t0 = d['timestamp'].min()
    d['t_min'] = (d['timestamp'] - t0).dt.total_seconds() / 60
    dock_frames.append(d[['timestamp','t_min','workload','run_id','name','cpu_pct','mem_mib']])
df_dock = pd.concat(dock_frames, ignore_index=True)

print(f'Summaries: {len(summaries)}  |  Batches: {len(df_tp)}  |  Docker rows: {len(df_dock):,}')
print('Containers:', sorted(df_dock['name'].unique()))

In [ ]:
# ── Tabela de summaries flat ──────────────────────────────────────────────────
rows = []
for s in summaries:
    pg, inf_, sc, bt = s['postgres'], s['influxdb'], s['stream_consumer'], s['batch_throughput']
    recv  = sc['contador']
    infl  = inf_['total_records']
    loss  = (recv - infl) / recv * 100 if recv else 0
    rows.append({
        'workload':          s['workload'],
        'run_id':            s['run_id'],
        'pg_records':        pg['total_records'],
        'batch_tp_rps':      bt['avg_throughput_records_per_s'],
        'lat_avg_s':         pg['latency_avg_ms'] / 1000,
        'lat_p50_s':         pg['latency_p50_ms'] / 1000,
        'lat_p95_s':         pg['latency_p95_ms'] / 1000,
        'stream_recv':       recv,
        'stream_influx':     infl,
        'stream_loss_pct':   round(loss, 2),
    })
df_sum = pd.DataFrame(rows)

print('⚠  4x R1 (jun-10): pg_records contém dados de run anterior (sem truncate).\n'
      '   Throughput batch e docker stats são válidos; pg_records NÃO comparável.\n')

display(
    df_sum.set_index(['workload','run_id'])
    .style
    .format({'pg_records':'{:,.0f}','batch_tp_rps':'{:,.0f}',
             'lat_avg_s':'{:.1f}','lat_p50_s':'{:.1f}','lat_p95_s':'{:.1f}',
             'stream_recv':'{:,.0f}','stream_influx':'{:,.0f}','stream_loss_pct':'{:.1f}%'})
    .background_gradient(subset=['stream_loss_pct'], cmap='Reds')
)

---
## 2. Consistência Intra-Workload

Verifica se as repetições de cada workload produziram resultados estáveis.  
**CV ≤ 10%** é considerado aceitável em benchmarks de sistemas.

In [ ]:
# ── Coeficiente de Variação ───────────────────────────────────────────────────
CV_METRICS = {
    'batch_tp_rps':    'Throughput batch (rec/s)',
    'lat_avg_s':       'Latência média (s)',
    'lat_p50_s':       'Latência p50 (s)',
    'lat_p95_s':       'Latência p95 (s)',
    'stream_loss_pct': 'Perda stream (%)',
}

cv_rows = []
for wl, grp in df_sum.groupby('workload'):
    for col, label in CV_METRICS.items():
        vals = grp[col].dropna()
        mean = vals.mean()
        std  = vals.std() if len(vals) > 1 else 0
        cv   = std / mean * 100 if mean else np.nan
        cv_rows.append({'Workload': wl, 'Métrica': label,
                        'Média': mean, 'DP': std, 'CV (%)': cv})

df_cv = pd.DataFrame(cv_rows)
cv_pivot = df_cv.pivot(index='Métrica', columns='Workload', values='CV (%)')

print('Coeficiente de Variação — CV ≤ 10% indica alta consistência\n')
display(
    cv_pivot.style
    .format('{:.1f}%', na_rep='—')
    .background_gradient(cmap='RdYlGn_r', vmin=0, vmax=15)
)

In [ ]:
# ── Box plots — throughput batch por run ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=FIG2T, sharey=False)

for ax, wl in zip(axes, WORKLOADS):
    grp  = df_tp[df_tp['workload'] == wl]
    runs = sorted(grp['run_id'].unique())
    data = [grp[grp['run_id'] == r]['throughput_records_per_s'].values for r in runs]

    bp = ax.boxplot(data, tick_labels=runs, patch_artist=True, notch=False,
                    medianprops=dict(color='black', linewidth=1.5),
                    flierprops=dict(marker='o', markersize=2.5, alpha=0.4))
    for patch in bp['boxes']:
        patch.set(facecolor=C[wl], alpha=0.65)

    ax.set_title(f'Workload {wl}', fontweight='bold')
    ax.set_xlabel('Execução')
    if ax is axes[0]:
        ax.set_ylabel('Throughput batch (rec/s)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v/1000:.0f}k'))

fig.suptitle('Consistência Intra-workload — Throughput Batch por Execução',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/01_consistencia_throughput_batch.pdf')
plt.show()

In [ ]:
# ── Consistência Docker: CPU e Memória por run ────────────────────────────────
CTRS = ['batch_consumer', 'stream_consumer', 'mosquitto', 'postgres', 'influxdb']

df_dagg = (
    df_dock[df_dock['name'].isin(CTRS)]
    .groupby(['workload','run_id','name'])[['cpu_pct','mem_mib']]
    .mean().reset_index()
)

for metric, ylabel, title_sfx in [
    ('cpu_pct', 'CPU médio (%)',       'CPU Médio'),
    ('mem_mib', 'Memória média (MiB)', 'Memória Média'),
]:
    fig, axes = plt.subplots(1, 3, figsize=FIG2T, sharey=False)
    run_colors = [C['1x'], C['2x'], C['4x']]

    for ax, wl in zip(axes, WORKLOADS):
        grp   = df_dagg[df_dagg['workload'] == wl]
        pivot = grp.pivot(index='name', columns='run_id', values=metric).reindex(CTRS).dropna(how='all')
        if pivot.empty: continue
        pivot.plot(kind='bar', ax=ax, color=run_colors[:len(pivot.columns)],
                   alpha=0.85, edgecolor='white', width=0.7)
        ax.set_title(f'Workload {wl}', fontweight='bold')
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=30)
        if ax is axes[0]:
            ax.set_ylabel(ylabel)
        ax.legend(title='Run', fontsize=7)

    fig.suptitle(f'Consistência Intra-workload — {title_sfx} por Container',
                 fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(f'{FIGS_DIR}/02_consistencia_{metric}.pdf')
    plt.show()

---
## 3. Comparativo entre Workloads

Cada gráfico agrega as execuções válidas de cada workload como **média ± desvio padrão**.

In [ ]:
# ── Helpers de plot ───────────────────────────────────────────────────────────
xi = np.arange(len(WORKLOADS))

def agg(metric):
    """Retorna (means, stds) na ordem de WORKLOADS."""
    means, stds = [], []
    for wl in WORKLOADS:
        vals = df_sum[df_sum['workload'] == wl][metric].dropna()
        means.append(vals.mean())
        stds.append(vals.std() if len(vals) > 1 else 0)
    return np.array(means), np.array(stds)

def bars(ax, means, stds, ylabel='', title='', log=False, annot_fmt=None):
    ax.bar(xi, means, yerr=stds, capsize=4, width=0.55,
           color=[C[w] for w in WORKLOADS], alpha=0.85,
           error_kw=dict(elinewidth=1.2, ecolor='#333'),
           edgecolor='#333', linewidth=0.5)
    ax.set_xticks(xi); ax.set_xticklabels(WORKLOADS)
    ax.set_ylabel(ylabel); ax.set_title(title, fontweight='bold')
    if log: ax.set_yscale('log')
    if annot_fmt:
        for i, (m, s) in enumerate(zip(means, stds)):
            ax.text(i, m + s + m*0.02, annot_fmt(m), ha='center', fontsize=7)

In [ ]:
# ── 3.1 Throughput Batch ──────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG2)

# Violino — distribuição completa por batch individual
data_v = [df_tp[df_tp['workload'] == wl]['throughput_records_per_s'].values for wl in WORKLOADS]
vp = ax1.violinplot(data_v, positions=xi, showmedians=True, showextrema=True, widths=0.65)
for pc, wl in zip(vp['bodies'], WORKLOADS):
    pc.set(facecolor=C[wl], alpha=0.65)
vp['cmedians'].set_color('black')
ax1.set_xticks(xi); ax1.set_xticklabels(WORKLOADS)
ax1.set_ylabel('Throughput (rec/s)')
ax1.set_title('Distribuição por batch', fontweight='bold')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v/1000:.0f}k'))

# Barras — média das médias por run
m, s = agg('batch_tp_rps')
bars(ax2, m, s, ylabel='rec/s', title='Média ± DP entre execuções',
     annot_fmt=lambda v: f'{v/1000:.0f}k')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v/1000:.0f}k'))

fig.suptitle('Throughput de Ingestão Batch (PostgreSQL)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/03_throughput_batch.pdf')
plt.show()

In [ ]:
# ── 3.2 Latência Batch ────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG2)

# Barras agrupadas: avg / p50 / p95
lat_cols = [('lat_avg_s','Média','#4477AA'),
            ('lat_p50_s','p50',  '#228833'),
            ('lat_p95_s','p95',  '#EE6677')]
width = 0.22
offsets = [-width, 0, width]
for (col, label, color), off in zip(lat_cols, offsets):
    m, s = agg(col)
    ax1.bar(xi + off, m, width=width, label=label, color=color, alpha=0.85,
            yerr=s, capsize=3, error_kw=dict(elinewidth=1), edgecolor='#333', linewidth=0.4)
ax1.set_xticks(xi); ax1.set_xticklabels(WORKLOADS)
ax1.set_ylabel('Latência (s)')
ax1.set_title('Latência avg / p50 / p95', fontweight='bold')
ax1.legend(fontsize=7)

# Scatter por run + banda de erro
m_avg, s_avg = agg('lat_avg_s')
ax2.fill_between(range(3), m_avg - s_avg, m_avg + s_avg, alpha=0.15, color='gray')
ax2.plot(range(3), m_avg, 'k--', linewidth=1.3, label='Média', zorder=4)
mk = {'R1':'o','R2':'s','R3':'^'}
for s in summaries:
    xi_pos = WORKLOADS.index(s['workload'])
    ax2.scatter(xi_pos, s['postgres']['latency_avg_ms']/1000,
                color=C[s['workload']], marker=mk.get(s['run_id'],'o'),
                s=55, zorder=5, label=f"{s['workload']}-{s['run_id']}")
ax2.set_xticks(range(3)); ax2.set_xticklabels(WORKLOADS)
ax2.set_ylabel('Latência média (s)')
ax2.set_title('Valor por execução', fontweight='bold')
ax2.legend(fontsize=6.5, ncol=2)

fig.suptitle('Latência de Ingestão Batch (PostgreSQL)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/04_latencia_batch.pdf')
plt.show()

In [ ]:
# ── 3.3 Registros ingeridos — Batch vs Stream ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=FIG2T)

for ax, wl in zip(axes, WORKLOADS):
    grp  = [s for s in summaries if s['workload'] == wl]
    rids = [s['run_id'] for s in grp]
    pg_r = [s['postgres']['total_records']        for s in grp]
    sc_r = [s['stream_consumer']['contador']       for s in grp]
    if_r = [s['influxdb']['total_records']         for s in grp]

    xg = np.arange(len(rids))
    w  = 0.25
    ax.bar(xg - w, pg_r, width=w, label='Batch (PG)',        color=C['batch'],  alpha=0.85, edgecolor='#333')
    ax.bar(xg,     sc_r, width=w, label='Stream recebido',   color='#AAAAAA',   alpha=0.85, edgecolor='#333')
    ax.bar(xg + w, if_r, width=w, label='Stream → InfluxDB', color=C['stream'], alpha=0.85, edgecolor='#333')
    ax.set_title(f'Workload {wl}', fontweight='bold')
    ax.set_xticks(xg); ax.set_xticklabels(rids); ax.set_xlabel('Execução')
    if ax is axes[0]:
        ax.set_ylabel('Registros'); ax.legend(fontsize=7)
    fmt = lambda v: f'{v/1e6:.2f}M' if v >= 1e6 else f'{v/1e3:.0f}k'
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: fmt(v)))

    for i,(rc,ri) in enumerate(zip(sc_r, if_r)):
        loss = (rc - ri) / rc * 100 if rc else 0
        col  = 'red' if loss > 5 else ('orange' if loss > 1 else 'green')
        ax.annotate(f'{loss:.1f}%', xy=(xg[i]+w, ri),
                    xytext=(0,4), textcoords='offset points',
                    ha='center', fontsize=6.5, color=col, fontweight='bold')

fig.suptitle('Registros Ingeridos por Pipeline (Batch vs. Stream)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/05_registros_batch_vs_stream.pdf')
plt.show()
print('⚠  4x R1: pg_records contém dados de run anterior — não comparável diretamente.')

In [ ]:
# ── 3.4 Perda de dados Stream ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=FIG1)

m, s = agg('stream_loss_pct')
bars(ax, m, s, ylabel='Perda de dados (%)',
     title='Perda Stream: MQTT → InfluxDB',
     annot_fmt=lambda v: f'{v:.1f}%')
ax.set_ylim(bottom=0)
ax.axhline(0, color='black', linewidth=0.7)

plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/06_perda_stream.pdf')
plt.show()

In [ ]:
# ── 3.5 Evolução temporal do throughput batch ─────────────────────────────────
df_tps = df_tp.sort_values('batch_id').copy()
df_tps['pct'] = df_tps.groupby(['workload','run_id'])['batch_id'].transform(
    lambda s: (s.rank()-1) / (len(s)-1) * 100 if len(s)>1 else 0)

fig, axes = plt.subplots(1, 3, figsize=FIG2T, sharey=True)
for ax, wl in zip(axes, WORKLOADS):
    for rid, rgrp in df_tps[df_tps['workload']==wl].groupby('run_id'):
        sm = rgrp.set_index('pct')['throughput_records_per_s'].rolling(5, min_periods=1).mean()
        ax.plot(sm.index, sm.values, label=rid, linewidth=1.2, alpha=0.85)
    ax.set_title(f'Workload {wl}', fontweight='bold')
    ax.set_xlabel('Progresso (%)')
    if ax is axes[0]: ax.set_ylabel('Throughput (rec/s)')
    ax.legend(fontsize=7)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v/1000:.0f}k'))

fig.suptitle('Evolução Temporal do Throughput Batch (média móvel, janela=5)',
             fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/07_throughput_temporal.pdf')
plt.show()

In [ ]:
# ── 3.6 CPU por container e workload ─────────────────────────────────────────
CTRS_PLOT = ['batch_consumer','stream_consumer','mosquitto','postgres','influxdb']

df_cpu = (
    df_dock[df_dock['name'].isin(CTRS_PLOT)]
    .groupby(['workload','run_id','name'])['cpu_pct'].mean().reset_index()
    .groupby(['workload','name'])['cpu_pct']
    .agg(['mean','std']).reset_index()
)

fig, ax = plt.subplots(figsize=FIG2)
xc = np.arange(len(CTRS_PLOT))
w3 = 0.25
offs3 = [-w3, 0, w3]
for off, wl in zip(offs3, WORKLOADS):
    g = df_cpu[df_cpu['workload']==wl].set_index('name').reindex(CTRS_PLOT)
    ax.bar(xc + off, g['mean'].fillna(0), width=w3, label=wl,
           color=C[wl], alpha=0.85,
           yerr=g['std'].fillna(0), capsize=3,
           error_kw=dict(elinewidth=1), edgecolor='#333', linewidth=0.4)
ax.set_xticks(xc); ax.set_xticklabels(CTRS_PLOT, rotation=20, ha='right')
ax.set_ylabel('CPU médio (%)')
ax.set_title('Utilização de CPU por Container e Workload', fontweight='bold')
ax.legend(title='Workload', fontsize=8)
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/08_cpu_por_container.pdf')
plt.show()

In [ ]:
# ── 3.7 Memória por container e workload ─────────────────────────────────────
df_mem = (
    df_dock[df_dock['name'].isin(CTRS_PLOT)]
    .groupby(['workload','run_id','name'])['mem_mib'].mean().reset_index()
    .groupby(['workload','name'])['mem_mib']
    .agg(['mean','std']).reset_index()
)

all_vals = df_mem['mean'].dropna()
use_log  = (all_vals.max() / all_vals.min()) > 20 if all_vals.min() > 0 else False

fig, ax = plt.subplots(figsize=FIG2)
for off, wl in zip(offs3, WORKLOADS):
    g = df_mem[df_mem['workload']==wl].set_index('name').reindex(CTRS_PLOT)
    ax.bar(xc + off, g['mean'].fillna(0), width=w3, label=wl,
           color=C[wl], alpha=0.85,
           yerr=g['std'].fillna(0), capsize=3,
           error_kw=dict(elinewidth=1), edgecolor='#333', linewidth=0.4)
ax.set_xticks(xc); ax.set_xticklabels(CTRS_PLOT, rotation=20, ha='right')
ylabel = 'Memória média (MiB)' + (' — escala log' if use_log else '')
ax.set_ylabel(ylabel)
ax.set_title('Utilização de Memória por Container e Workload', fontweight='bold')
ax.legend(title='Workload', fontsize=8)
if use_log: ax.set_yscale('log')
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/09_memoria_por_container.pdf')
plt.show()

In [ ]:
# ── 3.8 CPU temporal — batch_consumer e stream_consumer ──────────────────────
CTRS_TS = ['batch_consumer', 'stream_consumer']
fig, axes = plt.subplots(len(CTRS_TS), 1, figsize=(7.16, 4.8), sharex=False)

for ax, ctr in zip(axes, CTRS_TS):
    for wl in WORKLOADS:
        sub = df_dock[(df_dock['name']==ctr) & (df_dock['workload']==wl)].copy()
        if sub.empty: continue
        sub['t_bin'] = sub['t_min'].round(0)
        ts  = sub.groupby('t_bin')['cpu_pct'].mean().rolling(3, min_periods=1).mean()
        std = sub.groupby('t_bin')['cpu_pct'].std().fillna(0).rolling(3, min_periods=1).mean()
        ax.plot(ts.index, ts.values, label=wl, color=C[wl], linewidth=1.2)
        ax.fill_between(ts.index, (ts - std).clip(0), ts + std, alpha=0.12, color=C[wl])
    ax.set_ylabel('CPU (%)')
    ax.set_title(ctr.replace('_',' ').title(), fontweight='bold')
    ax.legend(title='Workload', fontsize=7, loc='upper right')

axes[-1].set_xlabel('Tempo (min)')
fig.suptitle('CPU ao Longo do Tempo — agregado por workload (média ± DP entre runs)',
             fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/10_cpu_temporal.pdf')
plt.show()

---
## 4. Painel Comparativo Final

In [ ]:
# ── Painel 2×2 para artigo ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(7.16, 5.2))
ax = axes.flatten()

# (A) Throughput batch
m, s = agg('batch_tp_rps')
bars(ax[0], m, s, ylabel='rec/s', title='(A) Throughput Batch',
     annot_fmt=lambda v: f'{v/1000:.0f}k')
ax[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v/1000:.0f}k'))

# (B) Latência média batch
m, s = agg('lat_avg_s')
bars(ax[1], m, s, ylabel='s', title='(B) Latência Média Batch',
     annot_fmt=lambda v: f'{v:.0f}s')

# (C) Perda stream
m, s = agg('stream_loss_pct')
bars(ax[2], m, s, ylabel='%', title='(C) Perda Stream (MQTT→InfluxDB)',
     annot_fmt=lambda v: f'{v:.1f}%')
ax[2].set_ylim(bottom=0)

# (D) CPU médio — batch_consumer vs stream_consumer
axd = ax[3]
CTRS_D = ['batch_consumer','stream_consumer']
df_cpud = (
    df_dock[df_dock['name'].isin(CTRS_D)]
    .groupby(['workload','name'])['cpu_pct'].mean().reset_index()
)
xd, wd = np.arange(len(WORKLOADS)), 0.3
for off, (ctr, col) in zip([-wd/2, wd/2],
                            [('batch_consumer',  C['batch']),
                             ('stream_consumer', C['stream'])]):
    vals = [df_cpud[(df_cpud['workload']==wl)&(df_cpud['name']==ctr)]['cpu_pct']
                   .values for wl in WORKLOADS]
    vals = [v[0] if len(v) else 0 for v in vals]
    axd.bar(xd + off, vals, width=wd, label=ctr.replace('_',' '),
            color=col, alpha=0.85, edgecolor='#333', linewidth=0.4)
axd.set_xticks(xd); axd.set_xticklabels(WORKLOADS)
axd.set_ylabel('CPU médio (%)')
axd.set_title('(D) CPU dos Consumers', fontweight='bold')
axd.legend(fontsize=7)

fig.suptitle('Comparativo Batch vs. Stream — Métricas Principais', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/11_painel_comparativo.pdf')
plt.show()

---
## 5. Tabela Consolidada para o TCC

In [ ]:
def fmt_mean_std(vals):
    if len(vals) == 0: return '—'
    m = np.mean(vals)
    s = np.std(vals)
    return f'{m:.1f} ± {s:.1f}' if len(vals) > 1 else f'{m:.1f}'

tbl_rows = []
for wl in WORKLOADS:
    wl_s  = [s for s in summaries if s['workload']==wl]
    tp_v  = [s['batch_throughput']['avg_throughput_records_per_s'] for s in wl_s]
    la_v  = [s['postgres']['latency_avg_ms']/1000 for s in wl_s]
    p95_v = [s['postgres']['latency_p95_ms']/1000 for s in wl_s]
    lo_v  = df_sum[df_sum['workload']==wl]['stream_loss_pct'].tolist()

    cpu_b  = df_dock[(df_dock['workload']==wl)&(df_dock['name']=='batch_consumer') ]['cpu_pct'].mean()
    cpu_s  = df_dock[(df_dock['workload']==wl)&(df_dock['name']=='stream_consumer')]['cpu_pct'].mean()
    mem_b  = df_dock[(df_dock['workload']==wl)&(df_dock['name']=='batch_consumer') ]['mem_mib'].mean()
    mem_s  = df_dock[(df_dock['workload']==wl)&(df_dock['name']=='stream_consumer')]['mem_mib'].mean()

    tbl_rows.append({
        'Workload':                  wl,
        'Throughput batch (rec/s)':  fmt_mean_std([int(v) for v in tp_v]),
        'Latência avg batch (s)':    fmt_mean_std(la_v),
        'Latência p95 batch (s)':    fmt_mean_std(p95_v),
        'Perda stream (%)':          fmt_mean_std(lo_v),
        'CPU batch_consumer (%)':    f'{cpu_b:.2f}',
        'CPU stream_consumer (%)':   f'{cpu_s:.2f}',
        'Mem batch_consumer (MiB)':  f'{mem_b:.1f}',
        'Mem stream_consumer (MiB)': f'{mem_s:.1f}',
    })

df_tbl = pd.DataFrame(tbl_rows).set_index('Workload')
print('Tabela consolidada (média ± DP entre execuções)\n')
display(df_tbl.T)